# Analisis Efisiensi Operasional dan Struktur Biaya E-Commerce Indonesia (2023–2025)

## Latar Belakang dan Tujuan Analisis

Proyek ini bertujuan mengevaluasi efisiensi operasional dan struktur biaya (terutama logistik/ongkir) pada transaksi e-commerce Indonesia selama 24 bulan (Des 2023 – Nov 2025). Analisis akan berfokus pada:

1) Struktur biaya pengiriman (shipping cost structure)  
2) Efektivitas subsidi pengiriman (shipping subsidy effectiveness)  
3) Risiko pembatalan pesanan (cancellation risk)  
4) Segmentasi area tidak efisien (operational inefficiency segmentation)

Output akhir: insight + rekomendasi strategis berbasis data

In [1]:
#Import library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
# Setting tampilan
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Setting gaya visual
sns.set_theme(style="whitegrid")

## 1. Data Understanding
Pada bagian ini yang dilakukan meliputi:
- Load dataset
- Cek struktur kolom, tipe data, missing values, dan ringkasan statistik awal
- Memastikan periode waktu data sesuai (24 bulan)


### Load Dataset

In [3]:

url = "https://raw.githubusercontent.com/AlvitoDwiP/operasional_ecommers_indo/refs/heads/main/data/raw/all_months_clean.csv"
df = pd.read_csv(
    url,
    sep=";",               
    encoding="utf-8-sig",  
    engine="python"
)

print("Shape:", df.shape)
df.head()


Shape: (20848, 19)


,order_id,total_qty,total_weight_gr,total_returned_qty,Total Diskon,product_categories,num_product_categories,Status Pesanan,Alasan Pembatalan,Opsi Pengiriman,Metode Pembayaran,Kota/Kabupaten,Provinsi,Ongkos Kirim Dibayar oleh Pembeli,Estimasi Potongan Biaya Pengiriman,Total Pembayaran,Perkiraan Ongkos Kirim,Waktu Pesanan Dibuat,source_file
0,ORD_0000001,2,2000,0,0,Celengan,1,Selesai,NaN,Reguler (Cashless)-SPX Standard,Saldo ShopeePay,KOTA SERANG,BANTEN,0,10000,38300,10000,2024-04-01 00:15,AprilSales2024.xlsx
1,ORD_0000002,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA SEMARANG,JAWA TENGAH,0,14500,18576,14500,2024-04-01 01:47,AprilSales2024.xlsx
2,ORD_0000003,1,500,0,0,Celengan,1,Selesai,NaN,Hemat Kargo-SPX Hemat,SeaBank Bayar Instan,KAB. BOGOR,JAWA BARAT,0,8000,7069,8000,2024-04-01 04:25,AprilSales2024.xlsx
3,ORD_0000004,2,400,0,0,Mangkok Sambal / Saus,1,Selesai,NaN,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA JAMBI,JAMBI,0,20000,32200,20000,2024-04-01 04:41,AprilSales2024.xlsx
4,ORD_0000005,3,3600,0,0,"Keranjang, Other, Tempat Nasi",3,Batal,Dibatalkan oleh Pembeli. Alasan: Ubah Pesanan ...,Hemat Kargo-SPX Hemat,COD (Bayar di Tempat),KOTA TANGERANG,BANTEN,0,0,0,8000,2024-04-01 06:12,AprilSales2024.xlsx


### Cek struktur kolom, tipe data, missing values, dan ringkasan statistik awal

#### **Cek tipe data & info kolom**


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20848 entries, 0 to 20847
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   order_id                            20848 non-null  object
 1   total_qty                           20848 non-null  int64 
 2   total_weight_gr                     20848 non-null  int64 
 3   total_returned_qty                  20848 non-null  int64 
 4   Total Diskon                        20848 non-null  int64 
 5   product_categories                  20848 non-null  object
 6   num_product_categories              20848 non-null  int64 
 7   Status Pesanan                      20848 non-null  object
 8   Alasan Pembatalan                   2830 non-null   object
 9   Opsi Pengiriman                     20848 non-null  object
 10  Metode Pembayaran                   20848 non-null  object
 11  Kota/Kabupaten                      20848 non-null  ob

**Interpretasi:**
- Memastikan kolom numerik (ongkir, berat, qty, diskon) benar-benar terbaca sebagai angka.
- Memastikan kolom waktu (waktu pesanan) masih string atau sudah datetime (biasanya masih string, jadi nanti kita parsing).

Daftar nama kolom (biar kita 100% presisi)

In [5]:
df.columns.tolist()


['order_id',
 'total_qty',
 'total_weight_gr',
 'total_returned_qty',
 'Total Diskon',
 'product_categories',
 'num_product_categories',
 'Status Pesanan',
 'Alasan Pembatalan',
 'Opsi Pengiriman',
 'Metode Pembayaran',
 'Kota/Kabupaten',
 'Provinsi',
 'Ongkos Kirim Dibayar oleh Pembeli',
 'Estimasi Potongan Biaya Pengiriman',
 'Total Pembayaran',
 'Perkiraan Ongkos Kirim',
 'Waktu Pesanan Dibuat',
 'source_file']

**Interpretasi:**
Ini dipakai untuk memastikan nama kolom yang akan kita rename atau pakai di rumus tidak salah ketik.

#### **Cek Missing values**

In [6]:
missing = df.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
missing.head(20)


,missing_count,missing_pct
Alasan Pembatalan,18018,86.43
Waktu Pesanan Dibuat,1980,9.50
order_id,0,0.00
total_weight_gr,0,0.00
total_qty,0,0.00
product_categories,0,0.00
total_returned_qty,0,0.00
num_product_categories,0,0.00
Status Pesanan,0,0.00
Opsi Pengiriman,0,0.00


**Interpretasi:**
- Kita fokus ke missing di kolom kunci: waktu, status, ongkir, subsidi, berat, pembayaran, wilayah.
- Jika missing kecil → aman. Jika besar → nanti ditentukan strategi (isi 0 / drop / imputasi ringan).

#### **Cek duplikasi order_id (jika ada)**

In [7]:
if "order_id" in df.columns:
    print("Duplikat order_id:", df["order_id"].duplicated().sum())
else:
    print("Kolom order_id tidak ditemukan, skip cek duplikasi order_id.")


Duplikat order_id: 0


**Interpretasi:**
- Idealnya setiap baris 1 order → order_id tidak duplikat.
- Jika duplikat ada, artinya data mungkin belum full agregasi atau ada issue merging.

#### **Statistik deskriptif numerik**

In [8]:
df.describe(include="number").T


,count,mean,std,min,25%,50%,75%,max
total_qty,20848.0,2.560821,7.796763,1.0,1.0,1.0,2.0,256.0
total_weight_gr,20848.0,2004.529691,7106.357515,10.0,300.0,500.0,1600.0,375000.0
total_returned_qty,20848.0,0.017748,0.548656,0.0,0.0,0.0,0.0,70.0
Total Diskon,20848.0,405.199348,9784.017705,0.0,0.0,0.0,0.0,700000.0
num_product_categories,20848.0,1.112097,0.485361,1.0,1.0,1.0,1.0,11.0
Ongkos Kirim Dibayar oleh Pembeli,20848.0,4189.990695,13578.489678,0.0,0.0,0.0,3000.0,584000.0
Estimasi Potongan Biaya Pengiriman,20848.0,10722.539284,12672.471066,0.0,0.0,9500.0,15000.0,312000.0
Total Pembayaran,20848.0,50683.597036,144019.263221,0.0,13480.0,21800.0,41600.0,3403591.0
Perkiraan Ongkos Kirim,20848.0,18425.626343,23791.508665,1.0,8000.0,11000.0,20000.0,959200.0


- Untuk “sanity check”: tidak boleh ada ongkir negatif, berat negatif, qty negatif.
- Jika max terlalu ekstrem → nanti kita handle outlier (bukan dihapus langsung, tapi di-clip/winsorize untuk visualisasi).

## 2. Persiapan dan Transformasi Data (Preprocessing Data)

Pada tahap ini, kita melakukan transformasi ringan untuk memastikan dataset siap dianalisis secara bisnis. Fokus kita adalah:
- Menstandarkan nama kolom
- Mengonversi waktu menjadi format datetime
- Membuat fitur turunan penting (bulan, cancel flag, ongkir per kg, rasio subsidi)
- Melakukan validasi dasar kualitas data

### **Standarisasi nama kolom**

In [9]:
rename_map = {
    "Total Diskon": "total_diskon",
    "Status Pesanan": "status_pesanan",
    "Alasan Pembatalan": "alasan_pembatalan",
    "Opsi Pengiriman": "opsi_pengiriman",
    "Metode Pembayaran": "metode_pembayaran",
    "Kota/Kabupaten": "kota_kabupaten",
    "Provinsi": "provinsi",
    "Ongkos Kirim Dibayar oleh Pembeli": "ongkir_dibayar",
    "Estimasi Potongan Biaya Pengiriman": "subsidi_ongkir",
    "Total Pembayaran": "total_pembayaran",
    "Perkiraan Ongkos Kirim": "perkiraan_ongkir",
    "Waktu Pesanan Dibuat": "waktu_pesanan",
    "Total Berat (gr)": "total_berat_gr",
    "Total Qty": "total_qty"
}

df = df.rename(columns=rename_map)
df.columns.tolist()


['order_id',
 'total_qty',
 'total_weight_gr',
 'total_returned_qty',
 'total_diskon',
 'product_categories',
 'num_product_categories',
 'status_pesanan',
 'alasan_pembatalan',
 'opsi_pengiriman',
 'metode_pembayaran',
 'kota_kabupaten',
 'provinsi',
 'ongkir_dibayar',
 'subsidi_ongkir',
 'total_pembayaran',
 'perkiraan_ongkir',
 'waktu_pesanan',
 'source_file']

**Interpretasi:**
Nama kolom kini lebih ringkas dan konsisten dengan tujuan memudahkan penulisan kode dan mengurangi risiko kesalahan akibat typo atau spasi.

### Konversi Waktu dan Fitur Bulanan
Tujuan:

Mengubah kolom waktu menjadi format datetime agar bisa dianalisis secara tren bulanan.

In [10]:
df["waktu_pesanan"] = pd.to_datetime(df["waktu_pesanan"], errors="coerce")

df["tahun"] = df["waktu_pesanan"].dt.year
df["bulan"] = df["waktu_pesanan"].dt.to_period("M").astype(str)
df["tanggal"] = df["waktu_pesanan"].dt.date

df[["waktu_pesanan", "tahun", "bulan"]].head()


,waktu_pesanan,tahun,bulan
0,2024-04-01 00:15:00,2024.0,2024-04
1,2024-04-01 01:47:00,2024.0,2024-04
2,2024-04-01 04:25:00,2024.0,2024-04
3,2024-04-01 04:41:00,2024.0,2024-04
4,2024-04-01 06:12:00,2024.0,2024-04


**Interpretasi**:
Kolom waktu berhasil dikonversi menjadi datetime. Fitur bulan akan digunakan untuk agregasi tren bulanan, yang lebih stabil dibanding analisis harian.